# Athena emulator — app integration

You're wiring a backend service to Athena: submit queries on behalf
of users, poll executions to a terminal state, page results back,
cancel runaway work, and reuse stored statements. Each call the
service makes runs against the live stack and records a measured
capability row.

Stack (`docker-compose.yml`, network `mlops_net`): `athena`
(emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab`
(this notebook). Per ADR-0002, only Athena traffic goes to the
emulator; S3/Glue calls must still land on moto.


In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid
from dataclasses import dataclass

import boto3
from _evidence import persist, record, render
from _helpers import (
    client_error_of,
    client_error_status,
    delete_run_buckets,
    delete_run_databases,
    delete_workgroup_contents,
    error_code_status,
    object_keys_under,
    query_execution,
    run_workgroup_names,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb04- prefix (nb04_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb04-"
DATABASE_PREFIX = "nb04_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
workgroup = f"{ATHENA_RESOURCE_PREFIX}wg-{RUN_SUFFIX}"
# Model pattern [a-zA-Z_][a-zA-Z0-9_@:]{1,256} — no dash, no leading digit.
statement_name = f"nb04_stmt_{RUN_SUFFIX}"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})
# The service submits under a dedicated workgroup with enforced
# ResultConfiguration: artifacts land under results/ without repeating
# OutputLocation per StartQueryExecution call.
athena.create_work_group(
    Name=workgroup,
    Configuration={
        "EnforceWorkGroupConfiguration": True,
        "ResultConfiguration": {"OutputLocation": f"s3://{bucket}/results/"},
    },
)


@dataclass(frozen=True)
class QueryRun:
    """Terminal outcome of one StartQueryExecution → poll cycle."""

    query_id: str
    state: str
    latency_ms: float
    execution: dict[str, object]


def start_query(query: str, **overrides: object) -> str:
    """Start ``query`` in the run workgroup; return its QueryExecutionId."""
    request: dict[str, object] = {
        "QueryString": query,
        "QueryExecutionContext": {"Database": database},
        "WorkGroup": workgroup,
    }
    request.update(overrides)
    return athena.start_query_execution(**request)["QueryExecutionId"]


def wait_terminal(query_id: str, timeout_s: float = 90.0) -> dict[str, object]:
    """Poll GetQueryExecution until ``query_id`` leaves QUEUED/RUNNING."""
    deadline = time.monotonic() + timeout_s
    execution = query_execution(athena, query_id)
    while execution["Status"]["State"] in {"QUEUED", "RUNNING"}:
        if time.monotonic() > deadline:
            raise TimeoutError(
                f"{query_id} still {execution['Status']['State']} "
                f"after {timeout_s}s"
            )
        time.sleep(0.25)
        execution = query_execution(athena, query_id)
    return execution


def run_to_terminal(query: str, **overrides: object) -> QueryRun:
    """Start ``query`` and poll it to a terminal state."""
    started = time.monotonic()
    query_id = start_query(query, **overrides)
    execution = wait_terminal(query_id)
    return QueryRun(
        query_id=query_id,
        state=execution["Status"]["State"],
        latency_ms=(time.monotonic() - started) * 1000,
        execution=execution,
    )


print(f"bucket={bucket} database={database} workgroup={workgroup}")

bucket=nb04-4cebdaa42680 database=nb04_16c36348 workgroup=nb04-wg-16c36348


## Submit and poll

A service drives the async state machine (ADR-0009): `StartQueryExecution`
returns the id immediately, `GetQueryExecution` polls QUEUED/RUNNING to a
terminal state, and a DML SELECT reports the `{QueryID}.csv` artifact.


In [3]:
select_run = run_to_terminal(
    "SELECT * FROM (VALUES (1,'a'),(2,'b'),(3,'c'),(4,'d'),(5,'e')) t(n,s)"
)
assert select_run.state == "SUCCEEDED"
assert select_run.execution["StatementType"] == "DML"
assert select_run.execution["SubstatementType"] == "SELECT"
assert select_run.execution["QueryExecutionContext"]["Database"] == database
output_location = select_run.execution["ResultConfiguration"]["OutputLocation"]
assert output_location.endswith(f"{select_run.query_id}.csv")
assert f"results/{select_run.query_id}.csv" in object_keys_under(
    s3, bucket, "results/"
)

record(
    "start→poll→GetQueryExecution (SELECT) + .csv artifact",
    "PASS",
    f"{select_run.query_id} {select_run.state} DML/SELECT; "
    f"OutputLocation={output_location}",
    select_run.latency_ms,
)

Evidence(feature='start→poll→GetQueryExecution (SELECT) + .csv artifact', status='PASS', detail='f1888038-1a08-4fef-a4fb-28136725b757 SUCCEEDED DML/SELECT; OutputLocation=s3://nb04-4cebdaa42680/results/f1888038-1a08-4fef-a4fb-28136725b757.csv', latency_ms=280.49857600126415)

## Page results back

`GetQueryResults` puts the header row on page zero and hands out an opaque
`NextToken` per page; boto3's paginator merges the pages losslessly — the
path a service's result endpoint walks.


In [4]:
first_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id, MaxResults=2
)
first_rows = first_page["ResultSet"]["Rows"]
assert [c["VarCharValue"] for c in first_rows[0]["Data"]] == ["n", "s"]
assert len(first_rows) == 3  # header + two data rows
assert "NextToken" in first_page

middle_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id,
    MaxResults=2,
    NextToken=first_page["NextToken"],
)
middle_rows = middle_page["ResultSet"]["Rows"]
assert [c["VarCharValue"] for c in middle_rows[0]["Data"]] == ["3", "c"]
assert len(middle_rows) == 2
assert "NextToken" in middle_page

last_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id,
    MaxResults=2,
    NextToken=middle_page["NextToken"],
)
assert len(last_page["ResultSet"]["Rows"]) == 1
assert "NextToken" not in last_page

paged_rows = []
paginator = athena.get_paginator("get_query_results")
for page in paginator.paginate(
    QueryExecutionId=select_run.query_id,
    PaginationConfig={"PageSize": 2},
):
    paged_rows.extend(page["ResultSet"]["Rows"])
assert len(paged_rows) == 6  # header + five data rows
assert [
    c["Name"]
    for c in first_page["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]
] == ["n", "s"]

record(
    "GetQueryResults paginates (header on page 0, opaque NextToken)",
    "PASS",
    "3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly",
)

Evidence(feature='GetQueryResults paginates (header on page 0, opaque NextToken)', status='PASS', detail='3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly', latency_ms=None)

## Track executions

`ListQueryExecutions` pages the workgroup's ids newest-first;
`BatchGetQueryExecution` resolves the known ones and reports the rest as
`UnprocessedQueryExecutionIds` — the shape a status view consumes.


In [5]:
# A second execution gives the list probe two run-scoped rows.
second_run = run_to_terminal("SELECT 2 AS two")
assert second_run.state == "SUCCEEDED"

batch = athena.batch_get_query_execution(
    QueryExecutionIds=[
        select_run.query_id,
        "00000000-0000-0000-0000-000000000000",
    ]
)
assert {e["QueryExecutionId"] for e in batch["QueryExecutions"]} == {
    select_run.query_id
}
unprocessed = batch["UnprocessedQueryExecutionIds"]
assert [u["QueryExecutionId"] for u in unprocessed] == [
    "00000000-0000-0000-0000-000000000000"
]
assert unprocessed[0]["ErrorCode"] == "InvalidRequestException"

# Most-recent-first, MaxResults-bounded pages — the path wrangler's cache
# probe walks before batch_getting each ID.
seen_ids = []
token = None
while True:
    page_args: dict[str, object] = {
        "WorkGroup": workgroup,
        "MaxResults": 1,
    }
    if token is not None:
        page_args["NextToken"] = token
    page = athena.list_query_executions(**page_args)
    seen_ids.extend(page["QueryExecutionIds"])
    token = page.get("NextToken")
    if token is None:
        break
assert seen_ids == [second_run.query_id, select_run.query_id]

record(
    "batch_get/list executions (unprocessed IDs, paged, newest first)",
    "PASS",
    "unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated "
    f"{len(seen_ids)} executions newest-first",
)

Evidence(feature='batch_get/list executions (unprocessed IDs, paged, newest first)', status='PASS', detail='unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated 2 executions newest-first', latency_ms=None)

## Runtime statistics

`GetQueryRuntimeStatistics` reports the engine counters (timeline millis,
input bytes/rows) a service surfaces for a finished execution.


In [6]:
statistics = athena.get_query_runtime_statistics(
    QueryExecutionId=select_run.query_id
)["QueryRuntimeStatistics"]
engine_ms = statistics["Timeline"]["EngineExecutionTimeInMillis"]
input_bytes = statistics["Rows"]["InputBytes"]
assert isinstance(engine_ms, int)
assert isinstance(input_bytes, int)

record(
    "GetQueryRuntimeStatistics (Timeline + Rows shape)",
    "PASS",
    f"EngineExecutionTimeInMillis={engine_ms} InputBytes={input_bytes}",
)

Evidence(feature='GetQueryRuntimeStatistics (Timeline + Rows shape)', status='PASS', detail='EngineExecutionTimeInMillis=2 InputBytes=0', latency_ms=None)

## Cancel a runaway query

`StopQueryExecution` kills the in-flight Trino statement — the execution
lands on CANCELLED. On an already-terminal execution AWS answers a 200
no-op; both paths are measured.


In [7]:
# Trino caps sequence() at 10000 entries (measured in the CLI consumer
# suite), so three capped unnests cross-joined stay RUNNING long enough
# for a client round-trip to land the stop.
SLOW_QUERY = (
    "SELECT count(*) FROM unnest(sequence(1, 10000)) a "
    "CROSS JOIN unnest(sequence(1, 10000)) b "
    "CROSS JOIN unnest(sequence(1, 10000)) c"
)
slow_id = start_query(SLOW_QUERY)

athena.stop_query_execution(QueryExecutionId=slow_id)
stopped = wait_terminal(slow_id)
record(
    "StopQueryExecution → CANCELLED",
    "PASS" if stopped["Status"]["State"] == "CANCELLED" else "FAIL",
    f"{slow_id} {stopped['Status']['State']}",
)

feature = "stop on a terminal execution (AWS no-ops with 200)"
terminal_stop_error = client_error_of(
    lambda: athena.stop_query_execution(QueryExecutionId=select_run.query_id)
)
if terminal_stop_error is None:
    record(feature, "PASS", "200 no-op on a SUCCEEDED execution")
else:
    code, status = error_code_status(terminal_stop_error)
    record(
        feature,
        "FAIL",
        f"{code} ({status}) on a SUCCEEDED execution; "
        "real AWS answers a 200 no-op",
    )

## Named queries

Workgroup-scoped canned SQL: create/get/list with `MaxResults`/`NextToken`
paging, `BatchGetNamedQuery` surfacing unknowns as `UnprocessedNamedQueryIds`,
a deleted lookup answering `InvalidRequestException` 400.


In [8]:
named_first = athena.create_named_query(
    Name=f"nb04-first-{RUN_SUFFIX}",
    Database=database,
    QueryString="SELECT 1",
    WorkGroup=workgroup,
    Description="first",
)["NamedQueryId"]
named_second = athena.create_named_query(
    Name=f"nb04-second-{RUN_SUFFIX}",
    Database=database,
    QueryString="SELECT 2",
    WorkGroup=workgroup,
)["NamedQueryId"]

fetched = athena.get_named_query(NamedQueryId=named_first)["NamedQuery"]
assert fetched["Name"] == f"nb04-first-{RUN_SUFFIX}"
assert fetched["Database"] == database
assert fetched["QueryString"] == "SELECT 1"
assert fetched["WorkGroup"] == workgroup

page_one = athena.list_named_queries(WorkGroup=workgroup, MaxResults=1)
page_two = athena.list_named_queries(
    WorkGroup=workgroup, MaxResults=1, NextToken=page_one["NextToken"]
)
assert page_one["NamedQueryIds"] + page_two["NamedQueryIds"] == [
    named_first,
    named_second,
]
assert "NextToken" not in page_two

record(
    "named queries create/get/list (MaxResults+NextToken pagination)",
    "PASS",
    f"2 queries in {workgroup}, first-page token={page_one['NextToken']!r}",
)

unknown_named_id = "00000000-0000-0000-0000-000000000000"
named_batch = athena.batch_get_named_query(
    NamedQueryIds=[named_first, unknown_named_id]
)
assert [q["NamedQueryId"] for q in named_batch["NamedQueries"]] == [
    named_first
]
assert named_batch["UnprocessedNamedQueryIds"] == [
    {"NamedQueryId": unknown_named_id}
]

athena.delete_named_query(NamedQueryId=named_second)
deleted_lookup = client_error_status(
    lambda: athena.get_named_query(NamedQueryId=named_second)
)
assert deleted_lookup == ("InvalidRequestException", 400)

record(
    "named queries batch_get unprocessed IDs + delete→400",
    "PASS",
    f"unprocessed={[unknown_named_id]}; deleted get → InvalidRequestException",
)

Evidence(feature='named queries batch_get unprocessed IDs + delete→400', status='PASS', detail="unprocessed=['00000000-0000-0000-0000-000000000000']; deleted get → InvalidRequestException", latency_ms=None)

## Prepared statements

Same workgroup-scoped CRUD surface plus `UpdatePreparedStatement` — the
store `EXECUTE` resolves against. Missing statements answer
`ResourceNotFoundException` 404, the contract wrangler's prepared path needs.


In [9]:
athena.create_prepared_statement(
    StatementName=statement_name,
    WorkGroup=workgroup,
    QueryStatement="SELECT ? AS chosen",
    Description="nb04 statement",
)
statement = athena.get_prepared_statement(
    StatementName=statement_name, WorkGroup=workgroup
)["PreparedStatement"]
assert statement["QueryStatement"] == "SELECT ? AS chosen"
assert statement["WorkGroupName"] == workgroup
assert "LastModifiedTime" in statement

athena.update_prepared_statement(
    StatementName=statement_name,
    WorkGroup=workgroup,
    QueryStatement="SELECT ? AS chosen /* v2 */",
    Description="nb04 statement v2",
)
updated = athena.get_prepared_statement(
    StatementName=statement_name, WorkGroup=workgroup
)["PreparedStatement"]
assert updated["QueryStatement"].endswith("/* v2 */")
assert updated["Description"] == "nb04 statement v2"

listed = athena.list_prepared_statements(WorkGroup=workgroup)[
    "PreparedStatements"
]
assert [s["StatementName"] for s in listed] == [statement_name]

statement_batch = athena.batch_get_prepared_statement(
    PreparedStatementNames=[statement_name, "nb04_missing"],
    WorkGroup=workgroup,
)
assert [s["StatementName"] for s in statement_batch["PreparedStatements"]] == [
    statement_name
]
assert statement_batch["UnprocessedPreparedStatementNames"] == [
    {"StatementName": "nb04_missing"}
]

record(
    "prepared statements create/get/update/list/batch_get",
    "PASS",
    f"{statement_name} in {workgroup}; unprocessed names surfaced",
)

missing_get = client_error_status(
    lambda: athena.get_prepared_statement(
        StatementName="nb04_missing", WorkGroup=workgroup
    )
)
missing_update = client_error_status(
    lambda: athena.update_prepared_statement(
        StatementName="nb04_missing",
        WorkGroup=workgroup,
        QueryStatement="SELECT 0",
    )
)
assert missing_get == ("ResourceNotFoundException", 404)
assert missing_update == ("ResourceNotFoundException", 404)

# The delete→404 probe burns a throwaway statement — the EXECUTE demo
# below still needs ``statement_name`` alive in the workgroup.
athena.create_prepared_statement(
    StatementName=f"{statement_name}_tmp",
    WorkGroup=workgroup,
    QueryStatement="SELECT 0",
)
athena.delete_prepared_statement(
    StatementName=f"{statement_name}_tmp", WorkGroup=workgroup
)
deleted_get = client_error_status(
    lambda: athena.get_prepared_statement(
        StatementName=f"{statement_name}_tmp", WorkGroup=workgroup
    )
)
assert deleted_get == ("ResourceNotFoundException", 404)

record(
    "prepared statements missing/deleted → ResourceNotFoundException 404",
    "PASS",
    "missing get/update + deleted get → 404",
)

Evidence(feature='prepared statements missing/deleted → ResourceNotFoundException 404', status='PASS', detail='missing get/update + deleted get → 404', latency_ms=None)

## EXECUTE with parameters

`EXECUTE <name>` resolves the stored statement server-side and binds
`ExecutionParameters` into its `?` markers (echoed on `GetQueryExecution`);
a bare `?` on a plain query takes the same binding. Resolution failures —
unknown name, `?`-count mismatch — land FAILED, never a submit-time 400.


In [10]:
exec_run = run_to_terminal(
    f"EXECUTE {statement_name}", ExecutionParameters=["hello"]
)
assert exec_run.state == "SUCCEEDED"
assert exec_run.execution["ExecutionParameters"] == ["hello"]
exec_rows = athena.get_query_results(QueryExecutionId=exec_run.query_id)[
    "ResultSet"
]["Rows"]
assert exec_rows[1]["Data"][0]["VarCharValue"] == "hello"
record(
    "EXECUTE + ExecutionParameters binds and echoes",
    "PASS",
    f"{statement_name} → row 'hello'; ExecutionParameters echoed",
    exec_run.latency_ms,
)

qmark_run = run_to_terminal("SELECT ? AS chosen", ExecutionParameters=["42"])
assert qmark_run.state == "SUCCEEDED"
qmark_rows = athena.get_query_results(QueryExecutionId=qmark_run.query_id)[
    "ResultSet"
]["Rows"]
assert qmark_rows[1]["Data"][0]["VarCharValue"] == "42"
record(
    "qmark ExecutionParameters on a plain query",
    "PASS",
    "SELECT ? with ['42'] → row '42'",
    qmark_run.latency_ms,
)

missing_stmt_run = run_to_terminal(f"EXECUTE {statement_name}_missing")
assert missing_stmt_run.state == "FAILED"
assert (
    "was not found in workGroup"
    in missing_stmt_run.execution["Status"]["StateChangeReason"]
)
mismatch_run = run_to_terminal(
    f"EXECUTE {statement_name}", ExecutionParameters=["a", "b"]
)
assert mismatch_run.state == "FAILED"
assert (
    "Incorrect number of parameters"
    in mismatch_run.execution["Status"]["StateChangeReason"]
)
record(
    "EXECUTE failures → FAILED not 400 (missing stmt, ?-count)",
    "PASS",
    "both FAILED with Athena's StateChangeReason phrasing",
)

Evidence(feature='EXECUTE failures → FAILED not 400 (missing stmt, ?-count)', status='PASS', detail="both FAILED with Athena's StateChangeReason phrasing", latency_ms=None)

## Idempotent and reusable submissions

`ClientRequestToken` dedupes a retried submit — the same token returns the
original QueryExecutionId. `ResultReuseConfiguration` re-answers a recent
identical query: `Statistics.ResultReuseInformation.ReusedPreviousResult`.


In [11]:
token = str(uuid.uuid4())
token_first = start_query("SELECT 1 AS one", ClientRequestToken=token)
token_second = start_query("SELECT 1 AS one", ClientRequestToken=token)
assert token_first == token_second
record(
    "ClientRequestToken dedupes retried submissions",
    "PASS",
    f"token {token[:8]}… → single execution {token_first[:8]}…",
)

reuse_configuration = {
    "ResultReuseByAgeConfiguration": {"Enabled": True, "MaxAgeInMinutes": 60}
}
reuse_first = run_to_terminal(
    "SELECT 7 AS seven", ResultReuseConfiguration=reuse_configuration
)
reuse_second = run_to_terminal(
    "SELECT 7 AS seven", ResultReuseConfiguration=reuse_configuration
)
assert reuse_second.state == "SUCCEEDED"
reused = (
    reuse_second.execution.get("Statistics", {})
    .get("ResultReuseInformation", {})
    .get("ReusedPreviousResult")
)
assert reused is True
assert "ResultReuseConfiguration" in reuse_second.execution
record(
    "ResultReuseConfiguration reuses recent results",
    "PASS",
    "second identical run reused (ReusedPreviousResult=True)",
)

Evidence(feature='ResultReuseConfiguration reuses recent results', status='PASS', detail='second identical run reused (ReusedPreviousResult=True)', latency_ms=None)

## Submission context

`ResultConfiguration.EncryptionConfiguration` echoes verbatim on
`GetQueryExecution` (the enforced workgroup's OutputLocation still wins);
`QueryExecutionContext.Catalog` names the registered catalog the execution
runs against.


In [12]:
encryption = {
    "EncryptionOption": "SSE_S3",
    "KmsKey": f"arn:aws:kms:us-east-1:123456789012:key/nb04-{RUN_SUFFIX}",
}
enc_run = run_to_terminal(
    "SELECT 1",
    ResultConfiguration={
        "OutputLocation": f"s3://{bucket}/enc/",
        "EncryptionConfiguration": encryption,
    },
)
assert enc_run.state == "SUCCEEDED"
reported = enc_run.execution["ResultConfiguration"]
# The workgroup is enforced, so its OutputLocation still wins; the
# request's EncryptionConfiguration is the member under test.
assert reported["EncryptionConfiguration"] == encryption
record(
    "ResultConfiguration.EncryptionConfiguration echoed verbatim",
    "PASS",
    "SSE_S3 + KmsKey echoed on GetQueryExecution",
    enc_run.latency_ms,
)

catalog_run = run_to_terminal(
    "SELECT 1",
    QueryExecutionContext={"Database": database, "Catalog": "AwsDataCatalog"},
)
assert catalog_run.state == "SUCCEEDED"
assert (
    catalog_run.execution["QueryExecutionContext"]["Catalog"]
    == "AwsDataCatalog"
)
record(
    "QueryExecutionContext.Catalog=AwsDataCatalog round-trips",
    "PASS",
    f"{catalog_run.query_id} {catalog_run.state}; catalog echoed",
    catalog_run.latency_ms,
)

Evidence(feature='QueryExecutionContext.Catalog=AwsDataCatalog round-trips', status='PASS', detail='d270b0ff-32cf-4525-93e5-bbe8b487e990 SUCCEEDED; catalog echoed', latency_ms=267.13931793347)

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/04_app_integration.md`; `PARITY.md` at the project root is
regenerated from all fragments.


In [13]:
print(render())
persist("04_app_integration")

| feature | status | detail | latency_ms |
|---|---|---|---|
| start→poll→GetQueryExecution (SELECT) + .csv artifact | PASS | f1888038-1a08-4fef-a4fb-28136725b757 SUCCEEDED DML/SELECT; OutputLocation=s3://nb04-4cebdaa42680/results/f1888038-1a08-4fef-a4fb-28136725b757.csv | 280.499 |
| GetQueryResults paginates (header on page 0, opaque NextToken) | PASS | 3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly |  |
| batch_get/list executions (unprocessed IDs, paged, newest first) | PASS | unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated 2 executions newest-first |  |
| GetQueryRuntimeStatistics (Timeline + Rows shape) | PASS | EngineExecutionTimeInMillis=2 InputBytes=0 |  |
| StopQueryExecution → CANCELLED | PASS | 6c8aacfb-74b9-422f-a171-5111724665d6 CANCELLED |  |
| stop on a terminal execution (AWS no-ops with 200) | PASS | 200 no-op on a SUCCEEDED execution |  |
| named queries create/get/list (MaxResults+NextToken pagination) | PASS | 2 queries i

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/04_app_integration.md')

## Clean up

The run's workgroup, bucket, and Glue database are throwaway — delete
them so the stack is left as you found it.


In [14]:
for name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
    delete_workgroup_contents(athena, name)
    athena.delete_work_group(WorkGroup=name)
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
print("cleaned: nb04 workgroups + buckets + glue databases removed")

cleaned: nb04 workgroups + buckets + glue databases removed
